###  ToolCallLimitMiddleware中间件
限制工具调用次数，可以 `限制所有工具` 调用的总次数，也可以 `限制特定工具` 的调用次数。
- 避免过多调用某些昂贵的外部API
- 限制网络爬虫或数据库查询请求的数量
- 避免Agent陷入无限循环
退出行为模式：
- error：直接抛异常
- end：结束整个会话
- continue：继续运行Agent，这是默认行为，此时Agent会将工具调用超出限制的信息传递给模
型，后者自主决定后续行为，如果模型能力不足，可能导致死循环，为了避免这种情况，我们实现
的fake server会以20%的概率输出正确响应，从而能终止循环。

###  ModelFallbackMiddleware中间件
用于故障转移，当主模型无法访问时，启用备用模型。

In [ ]:
# 1、模型的初始化
import os
from dotenv import load_dotenv
from langchain_qwq import ChatQwen

# 从.env文件中加载环境变量
load_dotenv(override=True)
# 模型的初始化
model = ChatQwen(
    model="qwen3.6-27b",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
)

In [ ]:
from langchain.agents.middleware import ModelFallbackMiddleware
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

# 定义主模型和备用模型
primary_model = init_chat_model("openai:gpt-5.4-mini")
fallback = ModelFallbackMiddleware(
fallback_models=[
    init_chat_model("openai:gpt-4o-mini"),
    init_chat_model("anthropic:claude-3-haiku")
]
)
agent = create_agent(
    model=primary_model,
    tools=[],
    middleware=[fallback],
)

###  LLMToolSelectorMiddleware中间件
智能工具筛选。
当工具太多时，用子模型筛选最相关的几个工具。

参数：
- model ：用于工具筛选的子模型
- max_tools ：限定可以调用的工具总数
- always_include ：指定的工具不被计数

###  ToolRetryMiddleware中间件
基于指数退避算法，设置工具调用失败时的重试策略。

指数退避（Exponential Backoff） 的核心思想就是：当某个操作失败（通常是网络请求、API 调
用或数据库连接）时，系统不会立刻重试，也不会每次都等待相同的固定时间，而是让每一次重试
的延迟时间按指数级增长。

jitter是为了避免大量工具的重试请求集中在固定的时间点，引入抖动。
假设：按照策略，两次工具调用请求的时间间隔应为10秒，加入抖动后，可能为8.9秒，也可能为10.2
秒。

In [ ]:
# 带抖动
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool
import datetime


def write_times(s):
    """将每次工具调用的时间戳和间隔写入本地文件，方便观察退避策略"""
    with open("call_times_with_jitter.txt", "a", encoding="utf-8") as f:
        f.write(s + "\n")


count = 1
start_time = None


@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    global start_time
    interval = 0
    current_time = datetime.datetime.now()
    if not start_time:
        interval = 0
    else:
        # 计算当前调用与上一次调用之间的时间差（秒）
        interval = (current_time - start_time).total_seconds()
    start_time = current_time
    res_str = (
        f"第 {count} 次调用，当前时间： {start_time}, "
        f"和上次调用间隔 {interval} 秒"
    )
    count += 1
    # 记录日志
    write_times(res_str)
    # 故意抛出 TimeoutError，以此触发中间件的重试机制
    raise TimeoutError("Not Implemented")


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        # ToolRetryMiddleware 用于捕获工具执行中的异常并自动重试
        ToolRetryMiddleware(
            max_retries=6,  # 最大重试次数（不包含初始的那次调用，一共最多调 1 + 6 = 7 次）
            backoff_factor=2.0,  # 指数退避因子（每次重试等待时间乘以 2）
            initial_delay=1.0,  # 第一次重试前的初始等待时间（1 秒）
            max_delay=10.0,  # 最大等待延迟上限（防止指数增长无限大，限制在 10 秒）
            jitter=True,  # 开启抖动（在等待时间中加入随机性，防止并发请求时出现“惊群效应”）
            retry_on=(TimeoutError,),  # 仅针对捕获到特定的 TimeoutError 异常时才触发重试
            on_failure="continue",  # 达到最大重试次数仍失败时，将错误信息塞回对话历史让模型继续决策
        ),
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("今天北京天气如何？")]
})

# 1. 你的提问 -> 2. AI 决定调用工具 -> 3. 重试失败后的错误反馈 -> 4. AI 最终给出的兜底回复
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

今天北京天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_e56cfea9afad47c591075c07)
 Call ID: call_e56cfea9afad47c591075c07
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

Tool 'get_weather' failed after 7 attempts with TimeoutError: Not Implemented. Please try again.
================================== Ai Message ==================================

很抱歉，我目前无法直接获取实时的天气信息。

建议您查看手机自带的天气应用或通过搜索引擎查询“北京今天天气”以获取最新的气象数据。如果您需要了解北京的气候特点或历史天气情况，我很乐意为您解答。


### ModelRetryMiddleware中间件

模型调用失败时重试，策略和工具调用的重试一样，都是基于指数退避算法。
因此，本节案例不再重点观察指数退避算法，而是测试不同的退出模式。

In [ ]:
# 继续运行
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware
from langchain.messages import HumanMessage
from dotenv import load_dotenv
load_dotenv(override=True)

agent = create_agent(
model="deepseek-cat",
middleware=[
ModelRetryMiddleware(
    max_retries=6,
    backoff_factor=2.0,
    initial_delay=1.0,
    max_delay=10.0,
    on_failure="continue",
    jitter=False,
),],)
response = agent.invoke({
    "messages": [HumanMessage("你好")]
})
for msg in response["messages"]:
    msg.pretty_print()

###  LLMToolEmulator中间件
某些情况下，工具尚未开发完成，我们希望先测试工具调用，可以用LLM tool emulator模拟工具。

In [5]:

from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator
from langchain.messages import HumanMessage
@tool
def get_weather(city: str):
    """查询指定城市天气"""
    return f"{city}今天天气晴朗"

agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[LLMToolEmulator(model=model,)]
)
response = agent.invoke({"messages": [HumanMessage("今天北京天气如何")]})
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_58ea6dd780d540ffb35a3d04)
 Call ID: call_58ea6dd780d540ffb35a3d04
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

{"city": "北京", "temperature": 19, "condition": "晴", "humidity": 42, "wind": "北风 3级", "aqi": 35}
================================== Ai Message ==================================

今天北京天气晴朗，气温约 19 度。

具体气象信息如下：
*   **天气状况**：晴
*   **气温**：19°C
*   **风力风向**：北风 3级
*   **湿度**：42%
*   **空气质量**：AQI 35（优）

天气状况良好，适合户外活动。


### ContextEditingMiddleware中间件
上下文编辑中间件，该中间件提供了上下文管理的一种方式。

通过更改发送给模型的消息列表来控制成本。

注意：不会更改消息列表。因此我们只能通过token用量来推测是否对消息列表进行了裁剪。

In [9]:
# 实验组-启用上下文编辑
from langchain.agents import create_agent
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit
from langchain.messages import HumanMessage, AIMessage
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

load_dotenv()

count = 0


@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    return (
        f"当前是第 {count} 次调用工具，{city}今天天气晴朗"
        f"天气非常好，北风，非常适合出行，盼望着，盼望着，"
        f"春天来了。我喜欢春天，你喜欢吗，天气真的很不错"
        f"万里无云，天气晴朗，春和景明，哈哈哈哈哈哈，这是凑字数的"
        f"真不错，天气非常好，适合出行，这里token挺多的"
        f"可以出门玩，尅有跑步，钓鱼，爬山，一切都很好哈哈哈"
    )


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(
                    trigger=50, # 触发阈值。token数或者消息数
                    keep=0, # 触发清理时，保留最近的几次工具调用
                ),
            ],
        ),
    ],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "1"}}

for i in range(3):
    print("=" * 30, f"当前是第 {i + 1} 轮调用", "=" * 30)
    count = i + 1
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    f"第 {i + 1} 次询问：今天北京天气如何，一句话回答"
                )
            ]
        },
        config=config,
    )
    print("---- 本次返回的 messages ----")
    for msg in response["messages"]:
        # msg.pretty_print()
        if isinstance(msg, AIMessage):
            if not msg.tool_calls:
                print(f"本次token用量：{msg.usage_metadata}")

============================== 当前是第 1 轮调用 ==============================


KeyboardInterrupt: 

1. `ContextEditingMiddleware` 的价值：大模型多轮对话时，如果频繁调用产生大量文本的工具
（如代码执行、网页爬取），历史记录会急剧膨胀。这个中间件就像一个“上下文抽脂手术”，在不
影响当前对话的前提下，自动在后台删掉之前沉淀的工具调用废话，从而极大地节省 Token 费用
并防止超出模型最大上下文窗口（Context Window）。

2. `InMemorySaver` ：它在内存中开辟了一个空间。第二轮和第三轮提问时，Agent 能通过
thread_id 自动找回前几轮的记忆。

###  FilesystemFileSearchMiddleware中间件
基于系统的Glob和Grep检索工具，为Agent赋予本地文件搜索和分析的能力。
- Glob根据文件路径检索
- Grep根据文件内容检索

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import FilesystemFileSearchMiddleware
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],  # 自动添加 Glob 和 Grep 工具
    middleware=[
        FilesystemFileSearchMiddleware(
            root_path="../todo_workspace",  # 搜索目录
            # 【可选】限制搜索的文件后缀，防止模型读取非代码或无关文件
            # allowed_extensions=[".py", ".ipynb", ".js", ".md"],  # 允许的文件类型
            # 是否启用 ripgrep 搜索引擎：
            # 设为 True 可以获得比原生 Grep 更快的性能（前提是系统已安装 ripgrep）
            use_ripgrep=True,
            # 单个文件的最大读取限制（单位MB）：防止读取超大型日志或二进制文件导致 OOM
            max_file_size_mb=10,
        ),
    ],
)

result = agent.invoke({
    "messages": [HumanMessage("找到包含add函数的Python或Jupyter文件")]
})


for msg in result["messages"]:
    msg.pretty_print()

### 3.10 Shell tool中间件
为Agent提供一个可以执行命令的Shell环境。
Windows下无法测试。
### 3.11 Filesystem中间件
这是源自deepagents（基于LangChain的另一个框架）的中间件
内置了四个工具，分别用于查看目录、读文件、写文件和改文件。
### 3.12 Subagent中间件
也是来自deepagents的中间件
用于便捷地创建子Agent。